# Notebook 2: Baseline Embeddings

**Goal:** Establish a baseline using pretrained sentence-transformer embeddings with the default text representation. Compare against a TF-IDF baseline.

**Research question:** How well do pretrained text embeddings capture food preferences compared to keyword-based (TF-IDF) retrieval?

**Evaluation protocol:** Item-to-item co-preference (see Section 3 below).

In [ ]:
# ============================================================
# SETUP
# ============================================================
import os, sys

REPO = "Embedding-Based-Recommender"
GITHUB_USER = "IldarRakiev"

ENV = 'kaggle' if os.path.exists('/kaggle/working') else 'colab'
BASE = '/kaggle/working' if ENV == 'kaggle' else '/content'
REPO_DIR = f'{BASE}/{REPO}'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/{GITHUB_USER}/{REPO}.git {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull -q')

os.system('pip install -q sentence-transformers faiss-cpu pandas pyarrow matplotlib seaborn umap-learn plotly scikit-learn tqdm')
sys.path.insert(0, f'{REPO_DIR}/src')
print(f"Environment: {ENV} | Repo: {REPO_DIR}")
print("Setup complete")

In [ ]:
# ============================================================
# DATA PATHS - auto-detected for Colab and Kaggle
# ============================================================
import os, glob as _glob

ENV = 'kaggle' if os.path.exists('/kaggle/working') else 'colab'

if ENV == 'kaggle':
    # PROCESSED_DIR: read-only input from NB01 output dataset
    _candidates = _glob.glob('/kaggle/input/**/processed/recipes.parquet', recursive=True)
    if _candidates:
        PROCESSED_DIR = os.path.dirname(_candidates[0])
        print(f"Reading from: {PROCESSED_DIR}")
    else:
        PROCESSED_DIR = '/kaggle/working/processed'
        print("NOTE: Add NB01 output as a dataset via 'Add Data'")
    # OUTPUT_DIR: writable working directory for saving results
    OUTPUT_DIR = '/kaggle/working/processed'
    os.makedirs(OUTPUT_DIR, exist_ok=True)
else:
    from google.colab import drive
    drive.mount('/content/drive')
    PROCESSED_DIR = '/content/drive/MyDrive/food-recsys/processed'
    OUTPUT_DIR = PROCESSED_DIR

print(f"OUTPUT_DIR  = {OUTPUT_DIR}")

In [ ]:
from text_builders import dish_to_rich_text
from embedding_model import EmbeddingModel
from utils import evaluate_all, plot_embeddings_umap, plot_nearest_neighbors
import pandas as pd
import numpy as np
np.random.seed(42)

In [ ]:

recipes = pd.read_parquet(f'{PROCESSED_DIR}/recipes.parquet')
train = pd.read_parquet(f'{PROCESSED_DIR}/interactions_train.parquet')
test = pd.read_parquet(f'{PROCESSED_DIR}/interactions_test.parquet')

print(f"Recipes: {len(recipes):,} | Train: {len(train):,} | Test: {len(test):,}")

## 1. Build Baseline Dish Texts

We use `dish_to_rich_text()` with **default flags** (baseline configuration):
- `include_recipe=True` — recipe instructions included
- `include_macro_tokens=True` — discrete HIGH_PROTEIN/LOW_FAT tokens included
- `include_ratios=False` — no continuous ratios
- `include_ingredients=False` — no extracted ingredient list

In [ ]:
# Build texts with baseline defaults (all flags at default)
texts = []
recipe_id_to_idx = {}
idx_to_recipe_id = {}

for i, (_, row) in enumerate(recipes.iterrows()):
    text = dish_to_rich_text(
        row.to_dict(),
        tags=row.get('tag_list', []),
        # All flags at default = baseline
    )
    texts.append(text)
    recipe_id_to_idx[row['id']] = i
    idx_to_recipe_id[i] = row['id']

print(f"Built {len(texts):,} texts")
print("\n=== Example text (index 0) ===")
print(texts[0][:500])

## 2. Generate Embeddings & Build FAISS Index

In [ ]:
import faiss

print("Loading embedding model...")
model = EmbeddingModel()  # paraphrase-multilingual-mpnet-base-v2, 768-dim
print(f"Model: {model.model_name} | dim={model.dim}")

print("\nEncoding texts...")
embeddings = model.encode(texts)  # shape: (N, 768), L2-normalized
print(f"Embeddings: {embeddings.shape} | {embeddings.nbytes / 1e6:.1f} MB")

# FAISS IndexFlatIP = exact cosine similarity (vectors are already normalized)
index = faiss.IndexFlatIP(model.dim)
index.add(embeddings)
print(f"FAISS index built: {index.ntotal:,} vectors")

## 3. Evaluation Protocol: Item-to-Item Co-Preference

Since Food.com has no user profiles, we evaluate through **co-preference**:
1. For each user in the test set with ≥5 positive ratings (≥4 stars)
2. Take one positive recipe as the **query**
3. The user's remaining positive recipes = **relevant set**
4. Search FAISS for top-K neighbors of the query
5. Measure: how many top-K results are in the relevant set?

This tests whether **recipes liked by the same person are nearby in embedding space** — the core assumption of content-based retrieval.

In [ ]:
def evaluate_retrieval(index, embeddings, recipe_id_to_idx, idx_to_recipe_id,
                       interactions_test, min_positives=5, ks=None):
    """Evaluate retrieval via item-to-item co-preference protocol."""
    if ks is None:
        ks = [5, 10, 20]

    user_positives = (
        interactions_test[interactions_test['rating'] >= 4]
        .groupby('user_id')['recipe_id']
        .apply(set)
        .to_dict()
    )

    results = []
    for user_id, pos_recipes in user_positives.items():
        # Keep only recipes that have embeddings
        pos_recipes = {r for r in pos_recipes if r in recipe_id_to_idx}
        if len(pos_recipes) < min_positives:
            continue

        pos_list = list(pos_recipes)
        # Use first positive as query, rest as relevant set
        query_recipe = pos_list[0]
        relevant = pos_recipes - {query_recipe}

        query_idx = recipe_id_to_idx[query_recipe]
        scores, indices = index.search(embeddings[query_idx:query_idx+1], max(ks) + 1)

        # Exclude the query itself
        recommended = [
            idx_to_recipe_id[i]
            for i in indices[0]
            if i >= 0 and idx_to_recipe_id.get(i) != query_recipe
        ]

        metrics = evaluate_all(recommended, relevant, ks=ks)
        results.append(metrics)

    if not results:
        return {}

    return pd.DataFrame(results).mean().to_dict()


print("Evaluating baseline embedding...")
baseline_metrics = evaluate_retrieval(index, embeddings, recipe_id_to_idx, idx_to_recipe_id, test)
print("\nBaseline Embedding Metrics:")
for k, v in sorted(baseline_metrics.items()):
    print(f"  {k}: {v:.4f}")

## 4. TF-IDF Baseline Comparison

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import scipy.sparse as sp

print("Fitting TF-IDF...")
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(texts)  # shape: (N, vocab)
print(f"TF-IDF matrix: {tfidf_matrix.shape}")

# Build a FAISS index from TF-IDF vectors (dense projection for comparison)
from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components=256, random_state=42)
tfidf_dense = svd.fit_transform(tfidf_matrix).astype(np.float32)
faiss.normalize_L2(tfidf_dense)

tfidf_index = faiss.IndexFlatIP(256)
tfidf_index.add(tfidf_dense)

print("Evaluating TF-IDF + SVD...")
tfidf_metrics = evaluate_retrieval(tfidf_index, tfidf_dense, recipe_id_to_idx, idx_to_recipe_id, test)

# Comparison table
import pandas as pd
comparison = pd.DataFrame({
    'TF-IDF + SVD(256)': tfidf_metrics,
    'Pretrained Embedding': baseline_metrics,
}).T
print("\n=== Comparison ===")
print(comparison.round(4))

## 5. Nearest Neighbor Sanity Check

In [ ]:
id_to_name = dict(zip(recipes['id'], recipes['name']))

# Pick 5 diverse query recipes
query_names = ['chocolate cake', 'grilled chicken', 'caesar salad', 'beef stew', 'banana bread']

for query_name in query_names:
    # Find closest recipe name
    match = recipes[recipes['name'].str.lower().str.contains(query_name, na=False)]
    if match.empty:
        continue
    query_id = match.iloc[0]['id']
    query_idx = recipe_id_to_idx.get(query_id)
    if query_idx is None:
        continue

    scores, indices = index.search(embeddings[query_idx:query_idx+1], 6)
    neighbor_names = [id_to_name.get(idx_to_recipe_id.get(i), '?') for i in indices[0][1:]]
    neighbor_scores = scores[0][1:].tolist()

    plot_nearest_neighbors(
        query_name=match.iloc[0]['name'],
        neighbor_names=neighbor_names,
        scores=neighbor_scores,
    )

## 6. UMAP Visualization

In [ ]:
# Sample for UMAP (full dataset would be slow)
SAMPLE = 3000
sample_idx = np.random.choice(len(embeddings), SAMPLE, replace=False)
sample_embs = embeddings[sample_idx]

# Color by meal-time tag
meal_tags_map = {'breakfast': 'breakfast', 'lunch': 'lunch', 'dinner': 'dinner', 'snacks': 'snack'}
sample_ids = [idx_to_recipe_id[i] for i in sample_idx]
id_to_tags = dict(zip(recipes['id'], recipes['tag_list']))

labels = []
for rid in sample_ids:
    tags = id_to_tags.get(rid, [])
    label = 'other'
    for tag in tags:
        if tag in meal_tags_map:
            label = meal_tags_map[tag]
            break
    labels.append(label)

plot_embeddings_umap(sample_embs, labels, title='Baseline Embeddings (colored by meal type)')

In [ ]:
# Save baseline results for Notebook 5
import json
all_results = {
    'tfidf_svd': tfidf_metrics,
    'baseline_embedding': baseline_metrics,
}
with open(f'{OUTPUT_DIR}/results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

# Save embeddings for reuse in NB3
np.save(f'{OUTPUT_DIR}/embeddings_baseline.npy', embeddings)
print("✓ Results and embeddings saved")